# Data preprocessing

In [1]:
# Loading librairies

import os
import re
import nltk
import spacy
import emoji
import datetime
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from datetime import date, timedelta

In [2]:
# Importation

df = pd.read_excel('scraped_data.xlsx', sheet_name='2018')
df.drop('Unnamed: 0', axis=1, inplace=True)
df.info()
df.sample(10, random_state = 123)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56219 entries, 0 to 56218
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       56219 non-null  datetime64[ns]
 1   Comment    56219 non-null  object        
 2   Reply      56219 non-null  int64         
 3   Retweet    56219 non-null  int64         
 4   Like       56219 non-null  int64         
 5   Quote      56219 non-null  int64         
 6   Creation   56219 non-null  datetime64[ns]
 7   Place      39387 non-null  object        
 8   Followers  56219 non-null  int64         
 9   Leader     56219 non-null  object        
dtypes: datetime64[ns](2), int64(5), object(3)
memory usage: 4.3+ MB


,Date,Comment,Reply,Retweet,Like,Quote,Creation,Place,Followers,Leader
27166,2018-12-10,Emmanuel RAMAZANI Shadary mon futur président....,0,0,0,0,2018-08-07,République Démocratique Du Con,0,Ramazani
30770,2018-12-12,@Fabiyo90 @tresorkikudi @StanysBujakera @Marti...,2,0,1,0,2018-11-14,"Paris, France",1137,Fayulu
7827,2018-11-26,CHAIRMAN @moise_katumbi VOUS VOULEZ FAIRE LES ...,0,0,0,0,2009-07-27,United Kingdom,27878,Fayulu
42405,2018-12-19,je vous dis mes três chers collegue dans son e...,0,0,0,0,2018-12-10,Brasil são paulo,0,Tshisekedi
55367,2018-12-30,Mon frère aller vote le numéro 4 pour le chang...,0,0,0,0,2018-03-22,Kinshasa,19,Fayulu
42647,2018-12-19,"Pendant que Félix Tshisekedi,Vital Kamerhe et ...",0,0,0,0,2011-02-15,Everywhere,157,Tshisekedi
35716,2018-12-15,@juliaferiol @StanysBujakera @MartinFayulu @Ad...,0,0,0,0,2010-10-25,"Kinshasa, Congo",769,Fayulu
18935,2018-12-05,@actualitecd Arrêter d ns metr d la poudr aux ...,0,0,0,0,2018-04-13,NaN,1,Ramazani
28168,2018-12-11,J'ai ajouté une vidéo à une liste de lecture @...,0,0,0,0,2017-08-16,kinshasa RDC,36,Fayulu
23879,2018-12-08,ET SI LE FCC JETE SON DÉVOLU SUR FÉLIX TSHISEK...,0,0,2,0,2017-03-27,NaN,70,Tshisekedi


In [3]:
# Converting emoji & emoticon to text data

df['Raw_Tweet'] = df['Comment'].astype(str).fillna("")
df['Comment'] = df['Comment'].apply(lambda t: emoji.demojize(t))

In [4]:
# Deleting hashtags and hyper-links

def clean_tweet(tweet):
    # hashtags and mentions
    tweet = re.sub(r'#\w+', '', tweet)
    tweet = re.sub('(@[A-Za-z0-9_]+)','', tweet)
    # hyper-links
    tweet = re.sub(r'http\S+', '', tweet)
    return tweet

df['Comment'] = df['Comment'].apply(lambda x: clean_tweet(x))

In [5]:
# Tokenization + Lemmatization using spaCy for french data

# !python -m spacy download fr_core_news_lg

nlp = spacy.load("fr_core_news_lg")     # or 'fr_core_news_md' (a medium model) / 'fr_core_news_sm' (a small model)
def spacy_clean(text):
    doc = nlp(text)
    toks = [token.lemma_.lower() for token in doc if not token.is_space and not token.is_punct]
    return toks

df['Comment'] = df['Comment'].apply(spacy_clean)

In [6]:
# Stopwords removal
relevant_stop_words = {"ne", "pas", "jamais", "plus"}
stop_words = set(stopwords.words('french')) - relevant_stop_words
df['Comment'] = df['Comment'].apply(lambda x: [word.lower() for word in x if word.lower() not in stop_words])

# Special characters handling
df['Comment'] = df['Comment'].apply(lambda x: [word for word in x if word.isalpha()])

In [7]:
# Viewing the dataframe after cleaning

df.sample(10, random_state = 123)

,Date,Comment,Reply,Retweet,Like,Quote,Creation,Place,Followers,Leader,Raw_Tweet
27166,2018-12-10,"[emmanuel, ramazani, shadary, futur, président...",0,0,0,0,2018-08-07,République Démocratique Du Con,0,Ramazani,Emmanuel RAMAZANI Shadary mon futur président....
30770,2018-12-12,"[frère, avoir, demander, pardon, retirer, paro...",2,0,1,0,2018-11-14,"Paris, France",1137,Fayulu,@Fabiyo90 @tresorkikudi @StanysBujakera @Marti...
7827,2018-11-26,"[chairman, vouloir, faire, election, deux, an,...",0,0,0,0,2009-07-27,United Kingdom,27878,Fayulu,CHAIRMAN @moise_katumbi VOUS VOULEZ FAIRE LES ...
42405,2018-12-19,"[dire, trê, cher, collegue, ensemble, candidat...",0,0,0,0,2018-12-10,Brasil são paulo,0,Tshisekedi,je vous dis mes três chers collegue dans son e...
55367,2018-12-30,"[frère, aller, vote, numéro, changement, pays,...",0,0,0,0,2018-03-22,Kinshasa,19,Fayulu,Mon frère aller vote le numéro 4 pour le chang...
42647,2018-12-19,"[pendant, félix, tshisekedi, vital, kamerhe, s...",0,0,0,0,2011-02-15,Everywhere,157,Tshisekedi,"Pendant que Félix Tshisekedi,Vital Kamerhe et ..."
35716,2018-12-15,"[quell, être, orientation, politique, fayulu, ...",0,0,0,0,2010-10-25,"Kinshasa, Congo",769,Fayulu,@juliaferiol @StanysBujakera @MartinFayulu @Ad...
18935,2018-12-05,"[arrêter, metr, poudr, oeil, ne, être, pa, rég...",0,0,0,0,2018-04-13,NaN,1,Ramazani,@actualitecd Arrêter d ns metr d la poudr aux ...
28168,2018-12-11,"[avoir, ajouter, vidéo, liste, lecture, réacti...",0,0,0,0,2017-08-16,kinshasa RDC,36,Fayulu,J'ai ajouté une vidéo à une liste de lecture @...
23879,2018-12-08,"[si, fcc, jete, dévolu, félix, tshisekedi, a, ...",0,0,2,0,2017-03-27,NaN,70,Tshisekedi,ET SI LE FCC JETE SON DÉVOLU SUR FÉLIX TSHISEK...


In [8]:
# Export and inspect
output_path = 'preprocessed_data.csv'
df.to_csv(output_path, index=False)
print(f"\nDataFrame successfully exported to: {output_path}")


DataFrame successfully exported to: preprocessed_data.csv
